# Inspect debug activation shards
Quick look at the output of `extract-windows --debug`.

In [ ]:
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer

OUT = Path('/orfeo/scratch/dssc/zenocosini/pile_gemma2b_activations_debug')
cfg = json.loads((OUT / 'config.json').read_text())
cfg

In [ ]:
shards = sorted((OUT / 'tokens').glob('shard_*.pt'))
print(f'{len(shards)} shards')
for p in shards:
    print(p.name)

In [ ]:
# Load shard 0
i = 0
tok = torch.load(OUT / 'tokens' / f'shard_{i:05d}.pt', weights_only=True)
a5  = torch.load(OUT / 'layer05' / f'shard_{i:05d}.pt', weights_only=True)
a17 = torch.load(OUT / 'layer17' / f'shard_{i:05d}.pt', weights_only=True)
meta = json.loads((OUT / 'meta' / f'shard_{i:05d}.json').read_text())
print('tokens :', tok.shape, tok.dtype)
print('layer5 :', a5.shape, a5.dtype)
print('layer17:', a17.shape, a17.dtype)
print('meta start/end :', meta['start'], meta['end'])
print('row_indices[:3]:', meta['row_indices'][:3])
print('rows[0]        :', meta['rows'][0])
print('finite :', torch.isfinite(a5).all().item(), torch.isfinite(a17).all().item())

In [ ]:
# Decode a window back to text
tokenizer = AutoTokenizer.from_pretrained(cfg['model'])
row = 0
r = meta['rows'][row]
print(f"dataset row {meta['row_indices'][row]}: subset={r['subset']}  "
      f"window=[{r['window_start']}:{r['window_end']}] / doc_len={r['doc_len']}")
print('---')
print(tokenizer.decode(tok[row].tolist()))

In [ ]:
# Norms across token positions — sanity check that early positions aren't broken
import matplotlib.pyplot as plt

norms5  = a5.float().norm(dim=-1).mean(dim=0)   # (window,)
norms17 = a17.float().norm(dim=-1).mean(dim=0)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(norms5, label='layer 5')
ax.plot(norms17, label='layer 17')
ax.axvline(cfg['drop_prefix'], color='k', ls='--', alpha=0.4, label=f"drop_prefix={cfg['drop_prefix']}")
ax.set_xlabel('position in window')
ax.set_ylabel('mean ‖x‖')
ax.legend(); fig.tight_layout()

In [ ]:
# Flatten into (N_tokens, d_model) dropping the under-contextualized prefix
P = cfg['drop_prefix']
X5  = a5[:, P:, :].reshape(-1, cfg['d_model'])
X17 = a17[:, P:, :].reshape(-1, cfg['d_model'])
print('X5 :', X5.shape, '  mean|x|:', X5.float().norm(dim=-1).mean().item())
print('X17:', X17.shape, '  mean|x|:', X17.float().norm(dim=-1).mean().item())